In [2]:
import torch.nn as nn
import torch

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        context_vectors = attn_weights @ values
        return context_vectors

In [4]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool() [: num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vectors = attn_weights @ values
        return context_vectors

In [19]:
input = torch.tensor([[1,2,3,4], [5,6,7,8], [9, 10,11,12]])
inputs = torch.stack((input, input))

In [21]:
inputs.shape # Batch, tokens, out_put

torch.Size([2, 3, 4])

In [23]:
inputs

tensor([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]],

        [[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]]])

In [27]:
inputs = inputs.view(2, 3, 2, 2)
inputs

tensor([[[[ 1,  2],
          [ 3,  4]],

         [[ 5,  6],
          [ 7,  8]],

         [[ 9, 10],
          [11, 12]]],


        [[[ 1,  2],
          [ 3,  4]],

         [[ 5,  6],
          [ 7,  8]],

         [[ 9, 10],
          [11, 12]]]])

In [28]:
inputs.transpose(1,2)

tensor([[[[ 1,  2],
          [ 5,  6],
          [ 9, 10]],

         [[ 3,  4],
          [ 7,  8],
          [11, 12]]],


        [[[ 1,  2],
          [ 5,  6],
          [ 9, 10]],

         [[ 3,  4],
          [ 7,  8],
          [11, 12]]]])

In [26]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out%num_heads == 0), "d_out must be divisible by num_heads"
        
        self.d_in = d_in
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = self.d_out // self.num_heads 
        self.contexr_length = context_length
        self.qkv_bias = qkv_bias
        
        self.W_query = nn.Linear(self.d_in, self.d_out, self.qkv_bias)
        self.W_key = nn.Linear(self.d_in, self.d_out, self.qkv_bias)
        self.W_value = nn.Linear(self.d_in, self.d_out, self.qkv_bias)
        
        self.out_proj = nn.Linear(self.d_out, self.d_out)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
                            "mask",
                            torch.triu(torch.ones(context_length, context_length),
                            diagonal=1)
                            )   
        
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x) # (b, num_tokens, dim_out)
        
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        
        queries = queries.transpose(1,2) # (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2)
        
        context_vec = context_vec.contiguous().view(
                                                    b, num_tokens, self.d_out
                                                    )
        context_vec = self.out_proj(context_vec)
        return context_vec
